In [1]:
import tfsolver
import numpy as np
import torch
import math

device = 'cpu'

# retrieve RESICE data
etd = torch.as_tensor(np.load('./RESICE/looker_etd_cached.npy')).to(device).to(torch.complex64).unsqueeze(2) #B,L,1,3,3
ltk = torch.as_tensor(np.load('./RESICE/looker_ltk_cached.npy')).to(device).to(torch.float32) * 100 + 1e-8 #B,L

#p_Euler = torch.zeros((1, ltk.size(1), 3), device=device, dtype=etd.dtype)
p_Euler = torch.randn((1, ltk.size(1), 3), device=device, dtype=ltk.dtype)
p_Euler[:,:,1:] = 0

p_Ni = torch.ones(1, device=device, dtype=etd.dtype)
p_Nt = p_Ni * 8.5 + 1j * 3.5

p_Theta = torch.ones(1, device=device, dtype=ltk.dtype)
wavelengths = torch.ones(1, device=device, dtype=ltk.dtype) * 22.9

coeffs = tfsolver.parallel_mm44(etd, p_Euler, ltk, p_Ni, p_Nt, p_Theta, wavelengths, device=device)

In [3]:
torch.finfo(torch.complex64).eps

1.1920928955078125e-07

In [232]:
def r4(x): return round(x.item().real, 2) + 1j *  round(x.item().imag, 2)
for i in range(20):
    print({'VV' : r4(coeffs['r_pp'][i]),
           'VH' : r4(coeffs['r_ps'][i]),
           'HV' : r4(coeffs['r_sp'][i]),
           'HH' : r4(coeffs['r_ss'][i])})

{'VV': (6.32-2.16j), 'VH': 0j, 'HV': 0j, 'HH': (-1.62+0.03j)}
{'VV': (-0.41+0.52j), 'VH': 0j, 'HV': 0j, 'HH': (-0.45-0.58j)}
{'VV': (1.15-0.15j), 'VH': (-0+0j), 'HV': (-0+0j), 'HH': (-1.05+0.08j)}
{'VV': (1.43+0.09j), 'VH': 0j, 'HV': 0j, 'HH': (-1.11-0.02j)}
{'VV': (0.64-0.81j), 'VH': (-0+0j), 'HV': (-0+0j), 'HH': (-0.92+0.45j)}
{'VV': (2.8+0.4j), 'VH': 0j, 'HV': 0j, 'HH': (-1.36-0.05j)}
{'VV': (1.63-2.85j), 'VH': 0j, 'HV': 0j, 'HH': (-1.82+0.29j)}
{'VV': (1.94-0.52j), 'VH': 0j, 'HV': 0j, 'HH': (-1.23+0.09j)}
{'VV': (2.41+0.32j), 'VH': 0j, 'HV': 0j, 'HH': (-1.27-0.07j)}
{'VV': (0.39-1.36j), 'VH': (-0+0j), 'HV': (-0+0j), 'HH': (-1.07+0.6j)}
{'VV': (1.54+0.56j), 'VH': 0j, 'HV': 0j, 'HH': (-1.15-0.11j)}
{'VV': (-0.49+0.29j), 'VH': (-0+0j), 'HV': (-0+0j), 'HH': (-0.24-0.46j)}
{'VV': (0.63+0.8j), 'VH': 0j, 'HV': 0j, 'HH': (-0.99-0.21j)}
{'VV': (-5.37-9.45j), 'VH': (-0+0j), 'HV': (-0+0j), 'HH': (-2.35+0.03j)}
{'VV': (1.8+0.02j), 'VH': 0j, 'HV': 0j, 'HH': (-1.19+0j)}
{'VV': (0.7-0.49j), 'VH':

In [130]:
torch.arange(etd.size(0))[coeffs['r_pp'].isnan().any(dim=-1).any(dim=-1)] #spot nulls
#etd[17]

tensor([17, 34, 41, 49, 61])

In [50]:
etd.unsqueeze(2).shape

torch.Size([149, 33, 1, 3, 3])

In [40]:
import tfsolver
import math
import torch

#:param N: Tensor
#PyTorch Tensor of shape [S x L x W x 3 x 3] with complex or real entries which contain the
#refractive index tensors for each layer of each thin film at the wavelengths of interest:
#S is the number of multi-layer thin films, L is the number of layers for each thin film, W is the number of
#wavelength considered.

device = 'cpu'

num_layers = 10

#fully isotropic media
p_N = (torch.linspace(2,3,num_layers,device=device)[None,:,None,None,None] * # rdie 2 to 3 with increasing depth
       torch.eye(3,3,device=device)[None,None,None,...]).to(torch.complex64) # idie 0.02 to 0.8 with increasing depth
p_N += (1j * torch.linspace(0.02,0.8,num_layers,device=device)[None,:,None,None,None] *
        torch.eye(3,3,device=device)[None,None,None,...])

# Random angular perturbations
#p_Euler = torch.zeros((1,num_layers,3),device=device)
p_Euler = torch.randn((1,num_layers,3),device=device)

# N layers with thicknesses 15 cm to 5 cm
p_D = torch.linspace(15,5,num_layers,device=device)[None,:]

# vacuum refractive index (incident medium)
p_Ni = torch.ones(1, device=device, dtype=torch.complex64)

# seawater refractive index (terminating medium)
p_Nt = torch.ones(1, device=device, dtype=torch.complex64) * 8.5 + 1j * 3.5

# incidence angle (22 - 24 degrees)
p_Theta = torch.linspace(math.radians(22), math.radians(24), 3, device=device)

# L-band @ 120 cm
wavelengths = torch.ones(1, device=device) * 120

coeffs = tfsolver.parallel_mm44(p_N, p_Euler, p_D, p_Ni, p_Nt, p_Theta, wavelengths, device=device)

In [38]:
{'VV': coeffs['r_pp'],'VH': coeffs['r_ps'],'HV':coeffs['r_sp'], 'HH': coeffs['r_ss']}

{'VV': tensor([[[0.3078-0.0084j],
          [0.3052-0.0087j],
          [0.3023-0.0090j]]]),
 'VH': tensor([[[1.0386e-08-2.6009e-09j],
          [1.0293e-08-2.6903e-09j],
          [1.0194e-08-2.7788e-09j]]]),
 'HV': tensor([[[-1.4917e-08+9.2102e-09j],
          [-1.4955e-08+9.3284e-09j],
          [-1.4986e-08+9.4501e-09j]]]),
 'HH': tensor([[[-0.3583+0.0086j],
          [-0.3605+0.0089j],
          [-0.3629+0.0092j]]])}